## Setup

- **API key**: The first code cell loads your OpenRouter API key from a `.env` file. The `.env` values replace any existing environment variables with the same name. If no key is found, the cell asks you to type it in. Never commit your key to Git.
- **Model**: The `MODEL` variable in `.env` chooses the model. If it is not set, the notebook uses `openai/gpt-6-luna`. Model names are listed at [openrouter.ai/models](https://openrouter.ai/models).

In [33]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv(override=True)

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter your OpenRouter API key: ")

MODEL = os.getenv("MODEL", "openai/gpt-6-luna")  # swap on openrouter.ai/models if needed

## Example selectors

A **few-shot prompt** shows the model a few worked examples before asking it a new question. It is like showing a new colleague three finished reports before asking them to write a fourth.

An **example selector** decides *which* examples go into the prompt. This matters when you have many examples. Including all of them can make the prompt too long, and the best examples are not always the first ones in the list.

LangChain (the library used in this notebook) provides these selectors:

- **Semantic similarity** (`SemanticSimilarityExampleSelector`): picks examples whose meaning is closest to the input.
- **Maximal marginal relevance** (`MaxMarginalRelevanceExampleSelector`): picks examples that are relevant *and* different from each other.
- **Length based** (`LengthBasedExampleSelector`): picks as many examples as fit in a length budget.
- **N-gram overlap** (`NGramOverlapExampleSelector`): picks examples that share words with the input.

Every selector has a `select_examples` method. It takes the user's input and returns a list of examples. Each selector decides for itself how to choose.

### The base interface

`BaseExampleSelector` is the base class that all selectors follow. It has two abstract methods, which means every subclass must write both:

- `add_example`: adds an example to the selector.
- `select_examples`: returns the examples to use for a given input.

This is a simplified view of the LangChain source:

```python
class BaseExampleSelector(ABC):
    @abstractmethod
    def add_example(self, example: Dict[str, str]) -> Any:
        """Add a new example to the selector."""

    @abstractmethod
    def select_examples(self, input_variables: Dict[str, str]) -> List[dict]:
        """Select which examples to use based on the inputs."""
```

### Build a custom example selector

The next cells build a simple selector that picks examples at random from a list. It ignores the input, so it is a useful starting point but not a smart selector.

In [34]:
from langchain_core.example_selectors.base import BaseExampleSelector
from typing import Dict, List
import random

class CustomExampleSelector(BaseExampleSelector):
    def __init__(self, examples: List[Dict[str, str]]):
        self.examples = examples

    def add_example(self, example: Dict[str, str]) -> None:
        """Add a new example to the list."""
        self.examples.append(example)

    def select_examples(self, input_variables: Dict[str, str]) -> List[dict]:
        """Randomly select examples based on the provided inputs."""
        return random.sample(self.examples, k=min(2, len(self.examples)))

In [35]:
examples = [
    {"recipe": "Spaghetti Carbonara"},
    {"recipe": "Chicken Alfredo"},
    {"recipe": "Vegetable Stir Fry"}
]

In [36]:
# initialize an example selector
example_selector = CustomExampleSelector(examples)

### Note: the input is not used here

The selector is called with `{"recipe": "recipe"}`. The word `recipe` is a placeholder, so its value does not matter. This selector ignores its input and picks at random. A smarter selector would use the input to choose examples.

In [37]:
# randomly select some examples
example_selector.select_examples({"recipe": "recipe"})

[{'recipe': 'Vegetable Stir Fry'}, {'recipe': 'Spaghetti Carbonara'}]

In [38]:
# Add a new recipe to the collection
example_selector.add_example({"recipe": "Beef Stroganoff"})
print(example_selector.examples)

[{'recipe': 'Spaghetti Carbonara'}, {'recipe': 'Chicken Alfredo'}, {'recipe': 'Vegetable Stir Fry'}, {'recipe': 'Beef Stroganoff'}]


In [39]:
# Randomly select two recipes again
example_selector.select_examples({"recipe": "recipe"})

[{'recipe': 'Chicken Alfredo'}, {'recipe': 'Spaghetti Carbonara'}]

## Select by length

This section uses LangChain's built-in `LengthBasedExampleSelector`. You do not need to write it yourself.

Why does length matter? A model can only read a limited amount of text in one request. This limit is called the **context window**. A prompt with many long examples can use up that space, so this selector keeps the examples within a set length budget.

In [40]:
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import FewShotPromptTemplate
from langchain_core.example_selectors import LengthBasedExampleSelector

examples = [
    {"input": "caffeinated", "output": "sleepy"},
    {"input": "spooky", "output": "cuddly"},
    {"input": "crispy", "output": "soggy"},
    {"input": "galactic", "output": "mundane"},
    {"input": "funky", "output": "plain-jane"},
]

In [41]:
example_prompt = PromptTemplate(
    input_variables=["input", "output"],
    template = "Input: {input}\nOutput: {output}"
)

### LengthBasedExampleSelector

The selector measures each example's length and keeps as many examples as fit in a budget called `max_length`.

**Length here means word count.** By default, the selector splits text on spaces and line breaks and counts the pieces. It does not count characters or model tokens.

How it works:

1. `add_example` formats the example with `example_prompt` and stores its word count.
2. `select_examples` counts the words in the input and subtracts them from `max_length`. What is left is the budget for examples.
3. It adds examples in order and stops at the first one that does not fit.

Shorter inputs leave more budget, so more examples are included. Longer inputs leave less, so fewer are included.

In the cells below, `max_length=15` means about 15 words. The very long input at the end of this section uses up the whole budget, so no examples are selected.

In [42]:
example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt= example_prompt,
    max_length=15,
)

In [43]:
dynamic_prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
    prefix="What's the opposite of...",
    suffix="Input: {adjective}\nOutput:",
    input_variables=["adjective"],
)

In [44]:
# Try changing the max length if no examples were selected

long_string = "fluffy and puffy and cloud-like and soft as a marshmallow and as cuddly as a teddy bear in a world of cotton candy"
print(dynamic_prompt.format(adjective=long_string))

What's the opposite of...

Input: fluffy and puffy and cloud-like and soft as a marshmallow and as cuddly as a teddy bear in a world of cotton candy
Output:


## Select by Maximal Marginal Relevance (MMR)

Maximal Marginal Relevance (MMR) is a method from information retrieval (finding useful results for a search). It balances two goals:

- **Relevance**: the example should be close in meaning to the input.
- **Diversity**: the example should not repeat what is already selected.

Without diversity, the top results can be near-copies of each other. If the three closest examples all say the same thing, the model sees little variety.

**Cosine similarity** measures how closely two lists of numbers point in the same direction. It ranges from -1 to 1, and values closer to 1 mean the texts are more alike. Embeddings, the lists of numbers used here, are explained in the section below.

### How the algorithm works

1. **Relevance**: calculate the cosine similarity between each example and the input.
2. **Diversity**: calculate how similar each example is to the examples already selected.
3. **Score**: combine the two:

$$ MMR = \lambda \times Relevance - (1 - \lambda) \times Diversity $$

Here `λ` (lambda) is between 0 and 1. A higher value favours relevance and a lower value favours diversity. The default in LangChain's MMR function is 0.5.

4. **Select**: pick the example with the highest score, add it to the selected set, and repeat for the remaining examples.
5. **Stop**: LangChain stops after it has picked `k` examples. It does not use a score threshold.

## What are embeddings?

An **embedding** is a list of numbers that represents the meaning of a piece of text. A short sentence might become a list of often over a thousand numbers.

The useful part is that texts with similar meanings get similar lists. A selector can compare these lists to find examples that are close in meaning, even when they share no words.

The next cell sets up the embeddings model. This notebook sends embedding requests through OpenRouter. The `langchain_openrouter` package only provides chat models, so the code uses LangChain's OpenAI embeddings class pointed at OpenRouter. Each embedding request is a network call and has a cost.

In [45]:
import os

from langchain_core.example_selectors import (
    MaxMarginalRelevanceExampleSelector,
    SemanticSimilarityExampleSelector,
)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_openrouter import ChatOpenRouter

# Embeddings go through OpenRouter. langchain_openrouter only provides chat models.
embeddings = OpenAIEmbeddings(
    model="openai/text-embedding-3-small",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    check_embedding_ctx_length=False,
)

In [46]:
examples

[{'input': 'caffeinated', 'output': 'sleepy'},
 {'input': 'spooky', 'output': 'cuddly'},
 {'input': 'crispy', 'output': 'soggy'},
 {'input': 'galactic', 'output': 'mundane'},
 {'input': 'funky', 'output': 'plain-jane'}]

In [47]:
example_prompt

PromptTemplate(input_variables=['input', 'output'], input_types={}, partial_variables={}, template='Input: {input}\nOutput: {output}')

### MaxMarginalRelevanceExampleSelector overview

`MaxMarginalRelevanceExampleSelector` uses MMR to pick examples that are both relevant and varied. Like `SemanticSimilarityExampleSelector`, it stores embeddings in a vector store and searches them.

**Key parts**

- **`fetch_k`** (default 20): how many candidates to fetch first. MMR then picks `k` of them.
- **`k`**: how many examples to return. The notebook sets `k=2`.
- **`from_examples(...)`**: builds the selector from a list of examples, an embeddings model, and a vector store class.
- **`select_examples(...)`**: fetches the `fetch_k` closest examples, then applies MMR to choose `k`.

**Reading the result (see the output of the `print` cell below)**

For the input "buzzed", the selector returns "caffeinated" and "funky". The first one picked is the closest match in meaning, because nothing has been selected yet to compare against. The second balances closeness with variety. Which examples win depends on the embeddings model, so a different model may choose differently.

In [48]:
mmr_selector = MaxMarginalRelevanceExampleSelector.from_examples(
    examples,
    embeddings,
    InMemoryVectorStore,
    k=2
)

In [49]:
mmr_prompt = FewShotPromptTemplate(
    # We provide an ExampleSelector instead of examples.
    example_selector=mmr_selector,
    example_prompt=example_prompt,
    prefix="What's the opposite of...",
    suffix="Input: {adjective}\nOutput:",
    input_variables=["adjective"],
)

In [50]:
print(mmr_prompt.format(adjective="buzzed"))

What's the opposite of...

Input: caffeinated
Output: sleepy

Input: funky
Output: plain-jane

Input: buzzed
Output:


## Select by n-gram overlap

The `NGramOverlapExampleSelector` ranks examples by how many words they share with the input.

An **n-gram** is a run of *n* words. A single word is a 1-gram, and two words in a row form a 2-gram. The selector scores overlap with a **BLEU score**. BLEU was designed to judge the quality of machine translation, and this selector reuses it to measure word overlap.

The score runs from 0.0 (no overlap) to 1.0 (full overlap).

The **threshold** sets a cut-off:

- **-1.0 (default):** keep every example, but sort them by score. This only reorders.
- **0.0:** sort the examples and drop any with no word overlap.
- **Above 1.0:** drop every example, so the result is an empty list.

> **Note:** this class lives in `langchain_community`. The deprecation warning shown by the next import cell says that package is being sunset, so the import path may change in future versions.

### NGramOverlapExampleSelector overview

**Key parts**

- **`examples`**: the examples the prompt template uses.
- **`example_prompt`**: the template that formats each example.
- **`threshold`**: the cut-off described above. Default -1.0.

**Matching is exact**

The selector splits the input and each example on spaces, then compares the words. This matching is:

- **Case-sensitive**: "Ghosted" and "ghosted" are different words.
- **Literal about punctuation**: `'Ghosted',` (with the quotes and comma) is a different word from `Ghosted`.

So the selector finds shared words, not shared meaning. The threshold 0.0 output below shows the effect.

**Methods**

- `add_example(example)`: adds an example to the list.
- `select_examples(input_variables)`: returns the examples sorted by overlap score, highest first. It drops any example at or below the threshold.

In [51]:
from langchain_community.example_selectors import NGramOverlapExampleSelector
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

In [52]:
# Examples of converting modern slang to Shakespearean English.
examples = [
    {"input": "He's flexing those dollar bills.", "output": "He doth flaunt his gold coins, verily."},
    {"input": "That's some tea.", "output": "That's a tale most intriguing."},
    {"input": "It's lit!", "output": "It's a merry revel!"},
    {"input": "Throwing shade", "output": "Casting aspersions"},
    {"input": "No cap", "output": "In sooth, no falsehood"},
    {"input": "Slide into the DMs", "output": "Venture into private missives"},
    {"input": "Ghosted", "output": "Vanish'd like a spectre"},
    {"input": "On fleek", "output": "In finest fettle"},
    {"input": "Bae", "output": "Mine own beloved"},
    {"input": "Squad goals", "output": "Band's aspirations"},
    {"input": "YOLO", "output": "Thou liv'st but once"},
    {"input": "FOMO", "output": "Fear of missing the revelry"},
    {"input": "Slay, queen!", "output": "Triumph, fair maiden!"},
    {"input": "I can't even", "output": "I am most perplexed"},
    {"input": "It's a vibe", "output": "It's a certain jest and merriment"},
    {"input": "Clap back", "output": "Retort with vigor"},
    {"input": "Low key", "output": "In hushed tones"},
    {"input": "High key", "output": "Loudly and proudly"},
    {"input": "Spill the tea", "output": "Unveil the tale"},
    {"input": "That's basic", "output": "That's most ordinary"},
    {"input": "Savage", "output": "Ruthless, like a wild beast"},
    {"input": "Mood", "output": "Mine current disposition"},
    {"input": "Woke", "output": "Awakened to the truths"},
    {"input": "Cancel culture", "output": "Banishment by the masses"},
    {"input": "Netflix and chill", "output": "Watch plays and relax, mayhaps more"},
    {"input": "Snack", "output": "A sight most pleasing"},
    {"input": "Thicc", "output": "Full and robust"},
    {"input": "Shook", "output": "Most startled and taken aback"},
    {"input": "AF", "output": "In great measure"}
]

### Why the second list uses full sentences

The cell above lists short slang terms with translations. The next cell replaces that list with full sentences that describe each slang term. The first list is not used after this point, so you can delete it to keep the notebook shorter.

The n-gram selector counts shared words. A sentence that names the slang term and explains it gives the selector more words to match. A short term such as "YOLO" gives it very little to work with.

In [53]:
examples = [

    {'input': "Saying 'He's flexing those dollar bills.' means he's showing off his wealth.",
     'output': 'To utter, "He doth flaunt his gold coins, verily." is to say he parades his riches.'},

    {'input': "The phrase 'That's some tea.' refers to juicy gossip or interesting news.",
     'output': "The saying, 'That's a tale most intriguing.' speaks of a story that piques interest."},

    {'input': "Exclaiming 'It's lit!' means the situation is exciting or fun.",
     'output': "Declaring 'It's a merry revel!' signifies a joyous occasion."},
    {'input': "When someone uses 'Throwing shade', they're subtly expressing disapproval or contempt.",
     'output': 'When one says "Casting aspersions", they art discreetly showing disdain or scorn.'},

    {'input': "The term 'No cap' is used to emphasize that someone is not lying.",
     'output': 'The phrase "In sooth, no falsehood" is uttered to stress that one speaks the truth.'},

    {'input': "To 'Slide into the DMs' means to send someone a direct message, usually with romantic intent.",
     'output': 'To "Venture into private missives" is to send a personal letter, perchance with courtly designs.'},

    {'input': "If someone has 'Ghosted', they've suddenly cut off all communication without explanation.",
     'output': "If one hath 'Vanish'd like a spectre', they've abruptly ceased all discourse without reason."},

    {'input': "The phrase 'On fleek' means that something is perfect or flawless.",
     'output': "The saying 'In finest fettle' signifies that something is in impeccable condition."}
]


In [54]:
example_prompt = PromptTemplate(
    input_variables=["input", "output"],
    template="Input: {input}\nOutput: {output}",
)

In [55]:
example_selector = NGramOverlapExampleSelector(
    examples=examples,
    example_prompt=example_prompt,
    threshold=-1.0
)

dynamic_prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
    prefix="Translate yon modern utterance into the tongue of the Bard",
    suffix="Modern: {sentence}\nShakespearean:",
    input_variables=["sentence"]
)


In [56]:
print(dynamic_prompt.format(sentence="Someone in a meeting mentioned they didn't want to be part of 'cancel culture'. What were they referring to?"))

Translate yon modern utterance into the tongue of the Bard

Input: To 'Slide into the DMs' means to send someone a direct message, usually with romantic intent.
Output: To "Venture into private missives" is to send a personal letter, perchance with courtly designs.

Input: The phrase 'That's some tea.' refers to juicy gossip or interesting news.
Output: The saying, 'That's a tale most intriguing.' speaks of a story that piques interest.

Input: The term 'No cap' is used to emphasize that someone is not lying.
Output: The phrase "In sooth, no falsehood" is uttered to stress that one speaks the truth.

Input: Saying 'He's flexing those dollar bills.' means he's showing off his wealth.
Output: To utter, "He doth flaunt his gold coins, verily." is to say he parades his riches.

Input: Exclaiming 'It's lit!' means the situation is exciting or fun.
Output: Declaring 'It's a merry revel!' signifies a joyous occasion.

Input: When someone uses 'Throwing shade', they're subtly expressing disapp

In [57]:
example_selector.threshold = 0.0
print(dynamic_prompt.format(sentence="The maiden ghosted me post our rendezvous."))

Translate yon modern utterance into the tongue of the Bard

Input: The phrase 'On fleek' means that something is perfect or flawless.
Output: The saying 'In finest fettle' signifies that something is in impeccable condition.

Input: The phrase 'That's some tea.' refers to juicy gossip or interesting news.
Output: The saying, 'That's a tale most intriguing.' speaks of a story that piques interest.

Input: The term 'No cap' is used to emphasize that someone is not lying.
Output: The phrase "In sooth, no falsehood" is uttered to stress that one speaks the truth.

Modern: The maiden ghosted me post our rendezvous.
Shakespearean:


### What the threshold did

- **Threshold -1.0 (cell 32):** all 8 examples stay in the prompt, sorted by overlap with the input. None is dropped.
- **Threshold 0.0 (cell 33):** only 3 examples remain, and each one shares the word "The" with the input. The "Ghosted" example is dropped even though the input contains "ghosted". The match is case-sensitive, and the example has `'Ghosted',` with punctuation attached.

The three examples kept share a common word, not a common meaning. This is the main limit of the n-gram selector.

## Select by semantic similarity

This selector finds examples whose **meaning** is close to the input, even when they share no words. It works with **embeddings**, the lists of numbers that represent meaning and that were explained earlier in this notebook.

### SemanticSimilarityExampleSelector overview

**Key parts**

- **`vectorstore`**: holds each example's embedding. This notebook uses `InMemoryVectorStore`, which keeps everything in memory.
- **`k`** (default 4): how many examples to return.
- **`example_keys`** (optional): only return these keys from each example.
- **`input_keys`** (optional): only use these input variables when searching.

**Methods**

- `add_example(example)`: embeds the example, stores it, and returns the ID of the stored text.
- `select_examples(input_variables)`: embeds the input, finds the `k` closest examples, and returns them.
- `from_examples(...)`: builds the selector from a list of examples, an embeddings model, and a vector store class.

**Always returns k examples**

The selector returns `k` examples every time, even when none is a good match. It has no built-in way to say "no good match". The outputs of the cells below show this.

In [58]:
examples

[{'input': "Saying 'He's flexing those dollar bills.' means he's showing off his wealth.",
  'output': 'To utter, "He doth flaunt his gold coins, verily." is to say he parades his riches.'},
 {'input': "The phrase 'That's some tea.' refers to juicy gossip or interesting news.",
  'output': "The saying, 'That's a tale most intriguing.' speaks of a story that piques interest."},
 {'input': "Exclaiming 'It's lit!' means the situation is exciting or fun.",
  'output': "Declaring 'It's a merry revel!' signifies a joyous occasion."},
 {'input': "When someone uses 'Throwing shade', they're subtly expressing disapproval or contempt.",
  'output': 'When one says "Casting aspersions", they art discreetly showing disdain or scorn.'},
 {'input': "The term 'No cap' is used to emphasize that someone is not lying.",
  'output': 'The phrase "In sooth, no falsehood" is uttered to stress that one speaks the truth.'},
 {'input': "To 'Slide into the DMs' means to send someone a direct message, usually wit

In [59]:
example_prompt

PromptTemplate(input_variables=['input', 'output'], input_types={}, partial_variables={}, template='Input: {input}\nOutput: {output}')

In [60]:
example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples,
    embeddings,
    InMemoryVectorStore,
    k=4
)
similar_prompt = FewShotPromptTemplate(
    # We provide an ExampleSelector instead of examples.
    example_selector=example_selector,
    example_prompt=example_prompt,
    prefix="Translate yon modern utterance into the tongue of the Bard",
    suffix="Modern: {sentence}\nShakespearean:",
    input_variables=["sentence"]
)

In [61]:
print(similar_prompt.format(sentence="What does it mean when someone says they're 'Throwing shade' at another?"))

Translate yon modern utterance into the tongue of the Bard

Input: When someone uses 'Throwing shade', they're subtly expressing disapproval or contempt.
Output: When one says "Casting aspersions", they art discreetly showing disdain or scorn.

Input: If someone has 'Ghosted', they've suddenly cut off all communication without explanation.
Output: If one hath 'Vanish'd like a spectre', they've abruptly ceased all discourse without reason.

Input: The phrase 'That's some tea.' refers to juicy gossip or interesting news.
Output: The saying, 'That's a tale most intriguing.' speaks of a story that piques interest.

Input: Exclaiming 'It's lit!' means the situation is exciting or fun.
Output: Declaring 'It's a merry revel!' signifies a joyous occasion.

Modern: What does it mean when someone says they're 'Throwing shade' at another?
Shakespearean:


In [62]:
print(similar_prompt.format(sentence="Someone told me my style was reminiscent of a 'snack'. What could they be hinting at?"))

Translate yon modern utterance into the tongue of the Bard

Input: The phrase 'That's some tea.' refers to juicy gossip or interesting news.
Output: The saying, 'That's a tale most intriguing.' speaks of a story that piques interest.

Input: When someone uses 'Throwing shade', they're subtly expressing disapproval or contempt.
Output: When one says "Casting aspersions", they art discreetly showing disdain or scorn.

Input: If someone has 'Ghosted', they've suddenly cut off all communication without explanation.
Output: If one hath 'Vanish'd like a spectre', they've abruptly ceased all discourse without reason.

Input: Saying 'He's flexing those dollar bills.' means he's showing off his wealth.
Output: To utter, "He doth flaunt his gold coins, verily." is to say he parades his riches.

Modern: Someone told me my style was reminiscent of a 'snack'. What could they be hinting at?
Shakespearean:


In [63]:
print(similar_prompt.format(sentence="I overheard someone mention they felt 'shook' after watching a movie. What emotion were they expressing?"))

Translate yon modern utterance into the tongue of the Bard

Input: If someone has 'Ghosted', they've suddenly cut off all communication without explanation.
Output: If one hath 'Vanish'd like a spectre', they've abruptly ceased all discourse without reason.

Input: When someone uses 'Throwing shade', they're subtly expressing disapproval or contempt.
Output: When one says "Casting aspersions", they art discreetly showing disdain or scorn.

Input: The phrase 'That's some tea.' refers to juicy gossip or interesting news.
Output: The saying, 'That's a tale most intriguing.' speaks of a story that piques interest.

Input: Exclaiming 'It's lit!' means the situation is exciting or fun.
Output: Declaring 'It's a merry revel!' signifies a joyous occasion.

Modern: I overheard someone mention they felt 'shook' after watching a movie. What emotion were they expressing?
Shakespearean:


### Limitation: no good match, still k results

In the two outputs above, the inputs mention "snack" and "shook". Neither word is in the 8-example list, so there is no good example to choose. The selector still returns 4 examples, and the model has to work with unrelated ones.

Compare this with the "Throwing shade" query earlier, which works well because that example is in the list.

Two lessons:

- A selector can only choose from the examples you give it. Adding examples for "snack" and "shook" would help.
- Check the selected examples, not just the final answer. A poor answer can start with poor examples.

## Sending the selected examples to a model

The code below connects three steps with `|` (the pipe symbol). The output of each step is passed into the next:

1. `similar_prompt`: fills the template with the selected examples and the new sentence.
2. `llm`: sends the prompt to the model through OpenRouter (`ChatOpenRouter`).
3. `StrOutputParser()`: turns the model's reply into plain text.

Things to notice in the output:

- **It translated the question rather than explaining it.** The prompt asks for a translation, so the model translated the whole sentence, including the question. It did not say what "Throwing shade" means. Prompt wording shapes the answer as much as the examples do.
- **It is one reply.** Another model, or the same model on another run, may word the answer differently. `temperature=0` reduces randomness but does not guarantee identical output.

In [64]:
llm = ChatOpenRouter(model=MODEL, temperature=0)
chain = similar_prompt | llm | StrOutputParser()
print(chain.invoke({"sentence": "What does it mean when someone says they're 'Throwing shade' at another?"}))

What doth it signify when one declares they are “casting aspersions” upon another?


## Responsible use and limits

- **Examples shape the output.** The examples a selector picks become part of the prompt, so they steer the model. Slang examples written by one person, or from one dialect, can skew results. Check whose language is represented.
- **Selectors cannot judge quality.** A selector ranks examples by words or meaning. It cannot tell whether an example is correct, fair, or up to date.
- **Data leaves your machine.** Embedding and chat requests are sent to OpenRouter, a third-party service. Do not send personal or confidential data without permission.
- **More examples cost more.** Each example adds tokens (the small pieces of text that models read and are billed for). Longer prompts can be slower and more expensive.